<a href="https://colab.research.google.com/github/Adrentur/-1_-_-_-3-10/blob/main/%D0%9B%D0%A0_9_%D0%9F%D0%BE%D0%BB%D1%94%D1%85%D1%96%D0%BD_3-10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** __________________________  
**Група:** _________________________________  
**Дата:** __________________________________

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.2.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
num_users = ratings_df['user_id'].nunique()
num_items = ratings_df['item_id'].nunique()
min_rating = ratings_df['rating'].min()
max_rating = ratings_df['rating'].max()
mean_rating = ratings_df['rating'].mean()
missing_values = ratings_df.isnull().sum().sum()

print(f"Кількість унікальних користувачів: {num_users}")
print(f"Кількість унікальних фільмів: {num_items}")
print(f"Рейтинг — мін: {min_rating}, макс: {max_rating}, середній: {mean_rating:.2f}")
print(f"Кількість пропущених значень: {missing_values}")

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Рейтинг — мін: 1.0, макс: 5.0, середній: 3.53
Кількість пропущених значень: 0


**Короткі спостереження щодо даних:**

про оцінки фільмів від тисяч користувачів у межах шкали від 1.0 до 5.0. Пропущені значення у вибірці відсутні, а розподіл оцінок демонструє, що користувачі найчастіше ставлять позитивні оцінки (переважно 3 або 4 бали).

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = sklearn_train_test_split(ratings_df, test_size=0.2, random_state=42)

print("Розмір навчальної вибірки:", train_df.shape)
print("Розмір тестової вибірки:", test_df.shape)

Розмір навчальної вибірки: (80000, 4)
Розмір тестової вибірки: (20000, 4)


In [6]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
reader = Reader(rating_scale=data.reader.rating_scale)

# Тільки необхідні колонки для train
train_data = Dataset.load_from_df(train_df[['user_id', 'item_id', 'rating']], reader)

# Формування тестового набору у вигляді списку кортежів (user_id, item_id, rating)
testset = list(test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None))

print("train_data підготовлено успішно. Кількість елементів у testset:", len(testset))

train_data підготовлено успішно. Кількість елементів у testset: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [7]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20],
    'min_k': [1],
    'sim_options': {
        'name': ['cosine'],
        'user_based': [False]
    }
}

**Обґрунтування вибору параметрів:**
Обґрунтування вибору параметрів (зразок):

Для SVD параметри n_factors (кількість латентних факторів), lr_all (швидкість навчання) та reg_all (коефіцієнт регуляризації) контролюють баланс між детальністю апроксимації матриці взаємодій та захистом від перенавчання.

Для KNNBasic кількість сусідів k, мінімальна кількість сусідів min_k та параметри подібності (cosine/pearson, а також user_based) визначають, наскільки релевантно формуються групи схожих користувачів чи об'єктів.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [8]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
grid_search_svd = GridSearchCV(SVD, param_grid_svd, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
grid_search_svd.fit(train_data)


In [9]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
grid_search_knn = GridSearchCV(KNNBasic, param_grid_knn, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
grid_search_knn.fit(train_data)

In [17]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
print("--- SVD Найкращі результати ---")
print("Найкращий RMSE (CV):", grid_search_svd.best_score['rmse'])
print("Найкращий MAE (CV):", grid_search_svd.best_score['mae'])
print("Найкращі параметри SVD:", grid_search_svd.best_params['rmse'])

print("\n--- KNNBasic Найкращі результати ---")
print("Найкращий RMSE (CV):", grid_search_knn.best_score['rmse'])
print("Найкращий MAE (CV):", grid_search_knn.best_score['mae'])
print("Найкращі параметри KNNBasic:", grid_search_knn.best_params['rmse'])

--- SVD Найкращі результати ---
Найкращий RMSE (CV): 0.9441985972079686
Найкращий MAE (CV): 0.7445252251190994
Найкращі параметри SVD: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

--- KNNBasic Найкращі результати ---
Найкращий RMSE (CV): 1.0723290336467126
Найкращий MAE (CV): 0.8541572225645183
Найкращі параметри KNNBasic: {'k': 20, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': False}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | ... | ... | ... |
| KNNBasic | ... | ... | ... |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [16]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

# TODO 6.2 — Створіть trainset із train_data
trainset = train_data.build_full_trainset()

# TODO 6.3 — Навчіть best_svd і best_knn із фіксацією random_state=42 для SVD
best_svd = SVD(**best_svd_params, random_state=42)
best_svd.fit(trainset)

best_knn = KNNBasic(**best_knn_params)
best_knn.fit(trainset)

print("Моделі успішно навчені на повній навчальній вибірці!")

Computing the cosine similarity matrix...
Done computing similarity matrix.
Моделі успішно навчені на повній навчальній вибірці!


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [18]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:
# Прогнозування на тестовому наборі
predictions_svd = best_svd.test(testset)
predictions_knn = best_knn.test(testset)

test_rmse_svd = accuracy.rmse(predictions_svd, verbose=False)
test_mae_svd = accuracy.mae(predictions_svd, verbose=False)

test_rmse_knn = accuracy.rmse(predictions_knn, verbose=False)
test_mae_knn = accuracy.mae(predictions_knn, verbose=False)

print(f"SVD Test RMSE: {test_rmse_svd:.4f} | MAE: {test_mae_svd:.4f}")
print(f"KNNBasic Test RMSE: {test_rmse_knn:.4f} | MAE: {test_mae_knn:.4f}")


SVD Test RMSE: 0.9192 | MAE: 0.7234
KNNBasic Test RMSE: 1.0439 | MAE: 0.8267


In [19]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:

results = pd.DataFrame([
    {"Model": "SVD", "RMSE": test_rmse_svd, "MAE": test_mae_svd},
    {"Model": "KNNBasic", "RMSE": test_rmse_knn, "MAE": test_mae_knn}
])

# Сортування за RMSE у зростаючому порядку (менше — краще)
results = results.sort_values(by="RMSE").reset_index(drop=True)

display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.043885,0.826673


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | ... | ... |
| KNNBasic | ... | ... |

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [20]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:
# Вибір моделі та користувача
target_model = best_svd  # Приклад обрання SVD за результатами кросвалідації
target_user_id = '196'

# Фільми, які користувач уже оцінив у навчальній вибірці
rated_items = set(
    trainset.to_raw_iid(inner_iid)
    for inner_iid, _ in trainset.ur[trainset.to_inner_uid(target_user_id)]
)

# Отримання списку всіх унікальних фільмів у trainset
all_item_raw_ids = [trainset.to_raw_iid(inner_iid) for inner_iid in trainset.all_items()]

# Фільтрація неоцінених фільмів та прогнозування
predictions = []
for item_id in all_item_raw_ids:
    if item_id not in rated_items:
        pred = target_model.predict(uid=target_user_id, iid=item_id)
        predictions.append((item_id, pred.est))

# Сортування за прогнозованим рейтингом у спадному порядку
predictions.sort(key=lambda x: x[1], reverse=True)
top_10 = predictions[:10]

print(f"Топ-10 рекомендацій для користувача {target_user_id}:")
for rank, (item_id, est_rating) in enumerate(top_10, 1):
    print(f"{rank}. Фільм ID: {item_id} | Прогнозований рейтинг: {est_rating:.2f}")

Топ-10 рекомендацій для користувача 196:
1. Фільм ID: 320 | Прогнозований рейтинг: 4.49
2. Фільм ID: 64 | Прогнозований рейтинг: 4.47
3. Фільм ID: 480 | Прогнозований рейтинг: 4.41
4. Фільм ID: 313 | Прогнозований рейтинг: 4.40
5. Фільм ID: 513 | Прогнозований рейтинг: 4.40
6. Фільм ID: 483 | Прогнозований рейтинг: 4.40
7. Фільм ID: 318 | Прогнозований рейтинг: 4.39
8. Фільм ID: 656 | Прогнозований рейтинг: 4.39
9. Фільм ID: 22 | Прогнозований рейтинг: 4.38
10. Фільм ID: 272 | Прогнозований рейтинг: 4.38


**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

## 9. Перевірка виконання

Позначте після реалізації:

- [+ ] Дані `ml-100k` завантажено та перевірено.
- [ +] Є розбиття 80/20 з фіксованим `random_state`.
- [ +] `Reader`, `train_data` і `testset` створені коректно.
- [+ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ +] Для SVD і KNNBasic показано параметри та результати CV.
- [ +] Моделі навчено на повній навчальній частині.
- [ +] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ +] Наведено порівняльну таблицю та інтерпретацію.
- [ +] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ +] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Яку модель обрано за кросвалідацією і чому?

За результатами крос-валідації було обрано модель SVD, оскільки вона продемонструвала нижчі показники помилок (менші значення RMSE та MAE) порівняно з KNNBasic. Факторний аналіз (латентні фактори) краще впорався з розрідженою матрицею оцінок фільмів.

Які найкращі гіперпараметри отримано для обох моделей?

Для SVD найкращими виявилися оптимальні значення кількості латентних факторів, швидкості навчання (lr_all) та регуляризації (reg_all). Для KNNBasic підібрано найкращі параметри кількості сусідів (
𝑘
k), мінімальної кількості сусідів (min_k) та обрану метрику схожості (косинусну міру / Pearson).

Які підсумкові значення RMSE та MAE на тестовій вибірці?

На незалежній тестовій вибірці модель SVD досягла кращих результатів (Test RMSE
≈
0.9192
≈0.9192, MAE
≈
0.7234
≈0.7234), тоді як модель KNNBasic показала дещо гіршу якість передбачення (Test RMSE
≈
1.0439
≈1.0439, MAE
≈
0.8267
≈0.8267).

Чи є різниця між результатами CV та тестування?

Отримані значення на незалежній тестовій вибірці є дуже близькими до результатів крос-валідації. Це свідчить про стабільність побудованих моделей та відсутність значного перенавчання (overfitting).

Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

Головним обмеженням отриманих моделей є «проблема холодного старту» (cold start). Система не може коректно генерувати персоналізовані рекомендації для нових користувачів або для нових фільмів, які не брали участі у навчанні через відсутність історії оцінок.

Підсумковий висновок: У ході виконання лабораторної роботи було успішно завантажено набір даних MovieLens 100K та побудовано дві рекомендаційні моделі — SVD і KNNBasic. За допомогою перехресної перевірки (GridSearchCV) було визначено оптимальні гіперпараметри, що дозволило налаштувати моделі для мінімізації похибок. Оцінювання на незалежній тестовій вибірці підтвердило перевагу моделі SVD (Test RMSE
≈
0.9192
≈0.9192), яка ефективніше виявляє приховані патерни уподобань користувачів порівняно з алгоритмом сусідства. На основі обраної моделі було успішно сформовано персоналізований топ-10 рекомендацій фільмів із виключенням раніше оцінених позицій. Отримані результати демонструють високу практичну ефективність коллаборативної фільтрації на базі матричної факторизації.

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.